In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning

In [2]:
print(conditioning.sample_riders(10, split="test"))
print(conditioning.sample_use_case(10, split="test"))
print(conditioning.sample_text(10, split="test"))

tensor([[0.3828, 0.4836, 0.6879, 0.4974, 0.3126, 0.3498],
        [0.4581, 0.4853, 0.6037, 0.5191, 0.3213, 0.3146],
        [0.4241, 0.5610, 0.6549, 0.5575, 0.3118, 0.3402],
        [0.3995, 0.4969, 0.6026, 0.4895, 0.3161, 0.3141],
        [0.4100, 0.4751, 0.6861, 0.5256, 0.2924, 0.3549],
        [0.4256, 0.5060, 0.6172, 0.5200, 0.3011, 0.3265],
        [0.4597, 0.4589, 0.6695, 0.5655, 0.3154, 0.3118],
        [0.4000, 0.5352, 0.6838, 0.5257, 0.3053, 0.2977],
        [0.4078, 0.5034, 0.6096, 0.5223, 0.2789, 0.3644],
        [0.3971, 0.5082, 0.6130, 0.5638, 0.3055, 0.3321]])
tensor([[0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 1., 0.]])
['Featuring fenders for added protection and disc wheels, the sky-blue Bolt Precision 610 is the perfect companion for your rides.\n', 'The light-periwinkle Nomad Impact-X includes a down tube-mount

In [24]:
data = pd.read_csv(split_datasets_path("CLIP_X_test.csv"), index_col=0)

data = data.sample(1024, random_state=0)

In [25]:
StandardEvaluations: List[EvaluationFunction] = [
    UsabilityEvaluator(),
    AeroEvaluator(),
    ErgonomicsEvaluator(),
    AestheticsEvaluator(mode="Text", batch_size=64),
    StructuralEvaluator(),
    ValidationEvaluator(),
    FrameValidityEvaluator()
]



evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [26]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")


condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [27]:
scores = evaluator(torch.tensor(data.values, dtype=torch.float32), condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [28]:
print(scores)

tensor([[   0.3264,   21.1613,    8.4912,  ...,  -38.6680, -109.4975,
           -0.5000],
        [   0.6808,   20.5413,   11.2713,  ...,   23.2218, -164.5000,
           -0.5000],
        [   0.5009,   22.9927,    1.9794,  ...,  -89.4056, -102.5000,
           -0.5000],
        ...,
        [   0.5240,   22.2174,    5.3247,  ...,  -65.0236,  -94.5000,
           -0.5000],
        [   0.5285,   22.7535,    6.5920,  ...,   -7.6727,  -94.5000,
           -0.5000],
        [   0.7326,   24.5551,   12.9578,  ...,  -50.9036, -112.5000,
           -0.5000]], grad_fn=<CopySlices>)


In [29]:
objective_scores = scores[:, isobjective]
constraint_scores = scores[:, ~isobjective]
print(objective_scores)
print(constraint_scores)

tensor([[ 0.3264, 21.1613,  8.4912,  ...,  2.8551,  2.2443,  2.0499],
        [ 0.6808, 20.5413, 11.2713,  ...,  1.7263,  2.3661,  1.9178],
        [ 0.5009, 22.9927,  1.9794,  ...,  2.7405,  1.5523,  1.7835],
        ...,
        [ 0.5240, 22.2174,  5.3247,  ...,  2.5563,  1.3644,  1.6569],
        [ 0.5285, 22.7535,  6.5920,  ...,  3.2875,  2.3791,  2.3105],
        [ 0.7326, 24.5551, 12.9578,  ...,  1.9031,  1.7487,  2.1199]],
       grad_fn=<IndexBackward0>)
tensor([[   1.3582,    0.9177, -160.0000,  ...,  -38.6680, -109.4975,
           -0.5000],
        [   0.9675,    0.7994, -188.2000,  ...,   23.2218, -164.5000,
           -0.5000],
        [   1.2870,    0.9051, -180.0000,  ...,  -89.4056, -102.5000,
           -0.5000],
        ...,
        [   1.2971,    0.9040, -163.3000,  ...,  -65.0236,  -94.5000,
           -0.5000],
        [   1.3377,    0.9784,  -12.1000,  ...,   -7.6727,  -94.5000,
           -0.5000],
        [   1.1492,    0.9985, -260.0000,  ...,  -50.9036, -112.5

In [ ]:
from pymoo.indicators.hv import HV

In [ ]:
#replace nan with -inf in objective_scores
objective_scores = scores[:, isobjective].detach().numpy()
constraint_scores = scores[:, ~isobjective].detach().numpy()

In [ ]:
validity_mask = np.all(constraint_scores <= 0, axis=1)

In [ ]:
constraint_scores.shape

(4512, 14)

In [ ]:
(constraint_scores <= 0).any(axis=0)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [ ]:
np.max(np.sum(constraint_scores <= 0, axis=1))

13

[[0.37109346 0.48904336 0.62334984 0.51484921 0.28319644 0.33466288]
 [0.44135011 0.51156794 0.68154445 0.54989605 0.27297983 0.37662222]
 [0.40613516 0.53210332 0.69825815 0.49177707 0.33684504 0.3250938 ]
 [0.37749998 0.53818009 0.60026729 0.51173852 0.30477969 0.31610153]
 [0.44109213 0.55698714 0.66609813 0.54763637 0.32788879 0.36390528]
 [0.42413773 0.56103382 0.6548594  0.55751451 0.3117992  0.34019701]
 [0.42762835 0.523274   0.59781853 0.50869598 0.2874827  0.34300858]
 [0.45636766 0.51517609 0.63939007 0.52355301 0.2602171  0.31698013]
 [0.38307297 0.49589817 0.63644308 0.52176332 0.33189469 0.36875567]
 [0.40882698 0.55270644 0.62772528 0.49036984 0.28552087 0.33429298]]
[[0 1 0]
 [0 0 1]
 [0 1 0]
 [1 0 0]
 [0 1 0]
 [0 1 0]
 [1 0 0]
 [0 0 1]
 [1 0 0]
 [0 1 0]]
['Designed with an enduring steel frame, the marine-blue Eagle Command promises unparalleled performance.\n'
 'The light-grey Viper Shift offers rugged trail bars to support your cycling ambitions.\n'
 'The red Tempest

In [ ]:
import pygmo as pg

def compute_ref_point(ref_scores):
    ref_scores[np.isnan(ref_scores)] = -float("inf")
    ref_point = np.max(ref_scores, axis=0)
    return ref_point

def hypervolume(objective_scores, constraint_scores):
    ref_point = compute_ref_point(objective_scores)

    validity_mask = np.all(constraint_scores <= 0, axis=1)
    valid_objective_scores = objective_scores[validity_mask]
    if valid_objective_scores.size == 0:
        return 0.0
    valid_objective_scores[np.isnan(valid_objective_scores)] = float("inf")
    valid_objective_scores = valid_objective_scores/ref_point
    valid_objective_scores = np.clip(valid_objective_scores, a_min=0, a_max=1)
    scaled_ref_point = np.ones_like(ref_point)

    hv = pg.hypervolume(valid_objective_scores)
    hv_value = hv.compute(ref_point=scaled_ref_point)
    return hv_value

print(hypervolume(objective_scores, np.zeros_like(constraint_scores)))

0.27726889436199187


In [ ]:
subset = objective_scores_capped[:1000,:]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ C:\Users\Lyle\AppData\Local\Temp\ipykernel_20704\2870073357.py:1 in <module>                     │
│                                                                                                  │
│ [Errno 2] No such file or directory:                                                             │
│ 'C:\\Users\\Lyle\\AppData\\Local\\Temp\\ipykernel_20704\\2870073357.py'                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'objective_scores_capped' is not defined

In [ ]:
subset.shape

(1000, 10)

In [ ]:
import pygmo as pg
hv = pg.hypervolume(objective_scores_capped/ref_point)
hv.compute(np.ones_like(ref_point))

0.22917476232497253